In [16]:
# Install dependencies if needed
# !pip install openai pypdf2 pydantic pandas python-dotenv

from openai import OpenAI
import os
import json
from pydantic import BaseModel, Field
from typing import List, Optional
import pandas as pd
from PyPDF2 import PdfReader
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Initialize the OpenAI client
# Assumes OPENAI_API_KEY is set in environment variables
client = OpenAI()

## Why OCR is Needed

Some PDF invoices are "image-based" (scanned documents or flattened images). In these cases, we need **Optical Character Recognition (OCR)** to convert the image of the text into actual string data before sending it to the LLM. We use `pdf2image` to convert PDF pages to images and `pytesseract` to read the text from those images.

In [17]:
# Import necessary libraries
from pdf2image import convert_from_path
import pytesseract
import sys

# Path to the PDF invoice file
INVOICE_PDF_PATH = "Dessert.Invoice.Example.pdf" 

# Explicitly set the tesseract executable path for macOS (Homebrew)
pytesseract.pytesseract.tesseract_cmd = r'/opt/homebrew/bin/tesseract'

# Convert each page of the PDF to an image
# Note: You may need to install poppler for pdf2image to work
# macOS: brew install poppler
# Linux: sudo apt-get install poppler-utils
# Windows: Download binary and add to PATH

# We explicitly point to the Homebrew poppler path for macOS
pages = convert_from_path(INVOICE_PDF_PATH, poppler_path="/opt/homebrew/bin")

invoice_text = ""
for page in pages:
    text = pytesseract.image_to_string(page)
    invoice_text += text + "\n"

print("--- OCR Extracted Text Preview ---")
print(invoice_text[:1500])

--- OCR Extracted Text Preview ---
INVOICE

FEASTFUL

caternc Dessert Cater ING *
Bill To:

John Doe, 456 Event Avenue, INV20240109
Townsville City, State, 12345 January 15, 2030
Assorted Desserts $20.00 $200.00
Customized Cake ] $80.00 $80.00
Dessert Buffet Setup 1 $150.00 $150.00
Delivery Charges 1 $25.00 $25.00

Subtotal $455.00
Terms and Conditions:
Tax (7%) $31.85

Payment is due within 15 days from

the date of the invoice. $486.85

Payment Method: Date: January 15, 2030
© Cash © Bank Transfer

© Debit ©) Digital wallet

Thank you once again for entrusting Michelle Ree,

us with your dessert catering. Manager

(123) 456-7890 Scanformore [l¥[=)
info@feastfulcatering.com information [ay yah





## 1. Raw Extraction

In this first approach, we simply ask the model to return a JSON object. We describe the fields we want, but we don't enforce a strict schema. The output is a raw JSON string, and the structure isn't guaranteed.

In [18]:
# Naive approach: Ask for JSON without a strict schema
naive_response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "system",
            "content": "You are a helpful assistant that extracts invoice data."
        },
        {
            "role": "user",
            "content": (
                "Read this commercial invoice and return a JSON object with "
                "vendor, invoice_number, invoice_date, total_amount, and "
                "line_items (description, hs_code, units, unit_price, line_total). "
                "Only return JSON, no explanation.\n\n"
                f"INVOICE TEXT:\n{invoice_text}"
            ),
        },
    ],
    response_format={"type": "json_object"}
)

# Extract the raw JSON string
naive_json = naive_response.choices[0].message.content
print("--- Raw JSON String ---")
print(naive_json)

# Parse it into a Python dictionary
parsed_naive = json.loads(naive_json)
parsed_naive

--- Raw JSON String ---
{
  "vendor": "FEASTFUL",
  "invoice_number": "INV20240109",
  "invoice_date": "January 15, 2030",
  "total_amount": 486.85,
  "line_items": [
    {
      "description": "Assorted Desserts",
      "hs_code": "",
      "units": 10,
      "unit_price": 20.00,
      "line_total": 200.00
    },
    {
      "description": "Customized Cake",
      "hs_code": "",
      "units": 1,
      "unit_price": 80.00,
      "line_total": 80.00
    },
    {
      "description": "Dessert Buffet Setup",
      "hs_code": "",
      "units": 1,
      "unit_price": 150.00,
      "line_total": 150.00
    },
    {
      "description": "Delivery Charges",
      "hs_code": "",
      "units": 1,
      "unit_price": 25.00,
      "line_total": 25.00
    }
  ]
}


{'vendor': 'FEASTFUL',
 'invoice_number': 'INV20240109',
 'invoice_date': 'January 15, 2030',
 'total_amount': 486.85,
 'line_items': [{'description': 'Assorted Desserts',
   'hs_code': '',
   'units': 10,
   'unit_price': 20.0,
   'line_total': 200.0},
  {'description': 'Customized Cake',
   'hs_code': '',
   'units': 1,
   'unit_price': 80.0,
   'line_total': 80.0},
  {'description': 'Dessert Buffet Setup',
   'hs_code': '',
   'units': 1,
   'unit_price': 150.0,
   'line_total': 150.0},
  {'description': 'Delivery Charges',
   'hs_code': '',
   'units': 1,
   'unit_price': 25.0,
   'line_total': 25.0}]}

> **Note:** While we got JSON back, the keys might vary between runs, and data types (like strings vs numbers) aren't strictly enforced.

## 2. Define Structured Schema

Now, we define a strict schema using Pydantic. This tells the LLM exactly what structure we expect, including data types and descriptions for each field.

In [19]:

class LineItem(BaseModel):
    description: str = Field(..., description="Description of the item")
    quantity: float = Field(..., description="Quantity from the Qty. column")
    unit_price: float = Field(..., description="Unit price (numeric, no currency symbol)")
    amount: float = Field(..., description="Line amount (numeric, no currency symbol)")

class Invoice(BaseModel):
    invoice_number: str = Field(..., description="Invoice number, e.g. INV20240109")
    bill_to_name: str = Field(..., description="Name under Bill To (e.g. John Doe)")
    bill_to_address: str = Field(..., description="Full address under Bill To, one string")
    invoice_date: str = Field(..., description="Invoice date as shown on the invoice")
    subtotal: float = Field(..., description="Subtotal before tax")
    tax_amount: float = Field(..., description="Tax amount")
    total_amount: float = Field(..., description="Total Amount on the invoice")
    line_items: List[LineItem] = Field(..., description="Line items from the description table")


## 3. Structured Extraction

We use the `client.beta.chat.completions.parse` method to get a strongly-typed `Invoice` object directly from the API. This ensures the output matches our Pydantic schema exactly.

In [20]:
structured_response = client.chat.completions.parse(
    model="gpt-4o-mini-2024-07-18",
    messages=[
        {
            "role": "system",
            "content": (
                "You extract structured data from invoices and MUST follow the "
                "Invoice Pydantic schema exactly.\n"
                "IMPORTANT RULES:\n"
                "- REMOVE currency symbols like '$' and commas.\n"
                "- RETURN numeric values as plain floats (e.g., 50.00 instead of $50.00).\n"
                "- DO NOT infer totals. Use ONLY values exactly shown in the invoice text.\n"
                "- DO NOT change quantities or dollar amounts.\n"
                "- If an amount says '$50.00', you MUST return: 50.00\n"
                "- Extract line items EXACTLY as shown in the table.\n"
            )
        },
        {
            "role": "user",
            "content": (
                "Extract the following fields from this catering invoice:\n\n"
                "invoice_number: text next to 'Invoice No' or 'INV'\n"
                "bill_to_name: name under 'Bill To:'\n"
                "bill_to_address: combine all address lines under Bill To\n"
                "invoice_date: the invoice date\n\n"
                "line_items: for each row in the table, extract:\n"
                "- description (text in Description column)\n"
                "- quantity (Qty column, as a number)\n"
                "- unit_price (numeric, REMOVE '$')\n"
                "- amount (numeric, REMOVE '$')\n\n"
                "summary fields:\n"
                "- subtotal (remove '$')\n"
                "- tax_amount (remove '$')\n"
                "- total_amount (remove '$')\n\n"
                "AGAIN: remove all currency symbols and do NOT infer values.\n\n"
                f"INVOICE TEXT:\n{invoice_text}"
            )
        }
    ],
    response_format=Invoice
)

invoice_obj = structured_response.choices[0].message.parsed
invoice_obj


Invoice(invoice_number='INV20240109', bill_to_name='John Doe', bill_to_address='456 Event Avenue, Townsville City, State, 12345', invoice_date='January 15, 2030', subtotal=455.0, tax_amount=31.85, total_amount=486.85, line_items=[LineItem(description='Assorted Desserts', quantity=1.0, unit_price=20.0, amount=200.0), LineItem(description='Customized Cake', quantity=1.0, unit_price=80.0, amount=80.0), LineItem(description='Dessert Buffet Setup', quantity=1.0, unit_price=150.0, amount=150.0), LineItem(description='Delivery Charges', quantity=1.0, unit_price=25.0, amount=25.0)])

In [21]:
# Keep the parsed invoice object in memory
parsed_invoice = invoice_obj

# Dict version (still useful to inspect in the notebook)
parsed_invoice_dict = parsed_invoice.model_dump()
print("--- Parsed Invoice Dictionary ---")
print(parsed_invoice_dict)

# Build DataFrame from line items
line_items_df = pd.DataFrame(
    [li.model_dump() for li in parsed_invoice.line_items]
)

# Ensure columns match the PDF table structure
# PDF headers: Description | Qty. | Unit Price | Amount
line_items_df = line_items_df[["description", "quantity", "unit_price", "amount"]]
line_items_df = line_items_df.rename(columns={
    "description": "Description",
    "quantity": "Qty",
    "unit_price": "Unit Price",
    "amount": "Amount",
})

print("\n--- Line Items DataFrame (matches PDF table) ---")
print(line_items_df)




--- Parsed Invoice Dictionary ---
{'invoice_number': 'INV20240109', 'bill_to_name': 'John Doe', 'bill_to_address': '456 Event Avenue, Townsville City, State, 12345', 'invoice_date': 'January 15, 2030', 'subtotal': 455.0, 'tax_amount': 31.85, 'total_amount': 486.85, 'line_items': [{'description': 'Assorted Desserts', 'quantity': 1.0, 'unit_price': 20.0, 'amount': 200.0}, {'description': 'Customized Cake', 'quantity': 1.0, 'unit_price': 80.0, 'amount': 80.0}, {'description': 'Dessert Buffet Setup', 'quantity': 1.0, 'unit_price': 150.0, 'amount': 150.0}, {'description': 'Delivery Charges', 'quantity': 1.0, 'unit_price': 25.0, 'amount': 25.0}]}

--- Line Items DataFrame (matches PDF table) ---
            Description  Qty  Unit Price  Amount
0     Assorted Desserts  1.0        20.0   200.0
1       Customized Cake  1.0        80.0    80.0
2  Dessert Buffet Setup  1.0       150.0   150.0
3      Delivery Charges  1.0        25.0    25.0


In [22]:
# Optionally save to CSV or Excel
# line_items_df.to_csv("invoice_line_items.csv", index=False)
# line_items_df.to_excel("invoice_line_items.xlsx", index=False)

## Next Steps

Now that we have a structured `Invoice` object, we can easily integrate this data into downstream systems:

*   **Database**: Insert the invoice header and line items into relational tables.
*   **ETL Pipeline**: Pass the structured object to a data pipeline for further processing.
*   **Analytics**: Join this data with other operational datasets for reporting.